In [2]:
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
###to read all pdfs inside the directory
def process(pdf_directory):
    #process all the pdf files creati ng an empty list
    all_documents=[]  # in this empty list  the total data  filesstored in the given path  should be 
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found{len(pdf_files)} pdf files to process")
    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            all_documents.extend(documents)
            print(f"loaded{len(documents)}pages")
        except Exception as e:
            print(f"error:{e}")
    print(f"\n total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents=process("./data")

found4 pdf files to process

processing:C Programming for absolute beginners.pdf
loaded335pages

processing:java for begginner.pdf
loaded148pages

processing:JS Notes.pdf
loaded45pages

processing:python.pdf
loaded164pages

 total documents loaded:692


In [4]:
print(len(all_pdf_documents))
all_pdf_documents

692


[Document(metadata={'producer': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creator': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creationdate': '2010-02-06T00:22:37+00:00', 'subject': 'Course Technology PTR', 'author': 'Michael Vine', 'codemantra, llc': 'http://www.codemantra.com', 'keywords': 'ISBN-13:    9781598634808', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 'moddate': '2010-09-10T17:08:32+03:00', 'title': 'C Programming for the Absolute Beginner, Second Edition', 'source': 'data\\C Programming for absolute beginners.pdf', 'total_pages': 335, 'page': 0, 'page_label': 'Cover'}, page_content=''),
 Document(metadata={'producer': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creator': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creationdate': '2010-02-06T00:22:37+00:00', 'subject': 'Course Technology PTR', 'aut

In [5]:
#chunking
def split_documnets(documents,chunk_size=300,chunk_overlap=200):
    """splitting documents for better rag experince"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""] 
        )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} into {len(split_docs)} pages")

    #example for print first 200 characters
    if split_docs:
        print(f"page content:{split_docs[0].page_content[:200]}...")
        print(f"meta data:{split_docs[0].metadata}")
    return split_docs


In [6]:
chunks=split_documnets(all_pdf_documents)
chunks

split 692 into 8131 pages
page content:C Programming
for the Absolute
Beginner,
Second Edition
MICHAEL VINE...
meta data:{'producer': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creator': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creationdate': '2010-02-06T00:22:37+00:00', 'subject': 'Course Technology PTR', 'author': 'Michael Vine', 'codemantra, llc': 'http://www.codemantra.com', 'keywords': 'ISBN-13:    9781598634808', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 'moddate': '2010-09-10T17:08:32+03:00', 'title': 'C Programming for the Absolute Beginner, Second Edition', 'source': 'data\\C Programming for absolute beginners.pdf', 'total_pages': 335, 'page': 1, 'page_label': 'i'}


[Document(metadata={'producer': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creator': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creationdate': '2010-02-06T00:22:37+00:00', 'subject': 'Course Technology PTR', 'author': 'Michael Vine', 'codemantra, llc': 'http://www.codemantra.com', 'keywords': 'ISBN-13:    9781598634808', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 'moddate': '2010-09-10T17:08:32+03:00', 'title': 'C Programming for the Absolute Beginner, Second Edition', 'source': 'data\\C Programming for absolute beginners.pdf', 'total_pages': 335, 'page': 1, 'page_label': 'i'}, page_content='C Programming\nfor the Absolute\nBeginner,\nSecond Edition\nMICHAEL VINE'),
 Document(metadata={'producer': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creator': 'Advanced PDF Repair at http://www.datanumen.com/apdfr/', 'creationdate': '

###embedding and vector db


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
from langchain_community.vectorstores import FAISS
vector_db=FAISS.from_documents(chunks,embeddings)

In [15]:
vector_db.save_local("faiss_index")

In [16]:
retreiver=vector_db.as_retriever()
doc2=retreiver.invoke("who invented python")
for doc in doc2:
    print(doc2)

[Document(id='ab89ef08-2b5b-4243-adbe-f2a07822fa6c', metadata={'producer': 'pdfTeX-1.10b', 'creator': 'TeX', 'creationdate': 'D:20050316115600', 'source': 'data\\python.pdf', 'total_pages': 164, 'page': 6, 'page_label': '7'}, page_content='for internet-based problems.\nPython was developed in the early 1990’s by Guido van Rossum, then\nat CWI in Amsterdam, and currently at CNRI in Virginia. In some ways,\npython grew out of a project to design a computer language which would be'), Document(id='46b05b21-6b41-4f65-b077-2cf08317b11f', metadata={'producer': 'pdfTeX-1.10b', 'creator': 'TeX', 'creationdate': 'D:20050316115600', 'source': 'data\\python.pdf', 'total_pages': 164, 'page': 6, 'page_label': '7'}, page_content='entirely in Java, further enhancing python’s position as an excellent solution\nfor internet-based problems.\nPython was developed in the early 1990’s by Guido van Rossum, then\nat CWI in Amsterdam, and currently at CNRI in Virginia. In some ways,'), Document(id='a98a627a-ef

In [19]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

In [21]:
context = "\n".join([doc.page_content for doc in doc2])

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question: who invented python
"""

response = llm.invoke(prompt)

print(response)

Guido van Rossum.
